1. Convert a 2D array into 1D using ravel.

2. Compare output of ravel() vs reshape(-1).

3. Modify a raveled array and observe original array.

4. Why is ravel() faster than flatten?

5. When should ravel() NOT be used?

6. Flatten a 3D array using ravel.

7. Explain memory sharing in ravel.

8. Create a matrix, ravel it, then reshape back.

9. Use ravel before applying vectorized operations.

10. Why analysts prefer ravel over loops?



In [1]:
import numpy as np 

In [2]:
arr = np.ones((3,3))
print(arr)
arr = arr.ravel()
print(arr)

[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]
[1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [3]:
2.
arr2 = np.array([[1,2,3],[4,5,6]])
print(arr2)
print()
arr2.ravel()

[[1 2 3]
 [4 5 6]]



array([1, 2, 3, 4, 5, 6])

In [4]:
arr2.reshape(-1)

array([1, 2, 3, 4, 5, 6])

In [5]:
3.
x = arr2.ravel()
print(x)
x[2 ] = 67
x[4 ] = 12
x[3 ] = 34
print(x)
print(arr2)

# we observe that when we modify the ravled array , the original array also changed because 
# it only biew the array and if we change the raveled array it will modify original array


[1 2 3 4 5 6]
[ 1  2 67 34 12  6]
[[ 1  2 67]
 [34 12  6]]


In [6]:
4. 
y = arr2.flatten()
y[0] = 99
print(y)
print(arr2)
# flatten method create a new array in the memory or copy of the array
# that is why it is slower then ravel

[99  2 67 34 12  6]
[[ 1  2 67]
 [34 12  6]]


Note : those method like reshape, ravel which use view to the array, if we want these method to copy so we can use .copy() method with them and it will copy the array 

**5. When should ravel() NOT be used?**


While `numpy.ravel()` is a fantastic tool for flattening arrays, it isn't a "one-size-fits-all" solution. Because it tries to return a **view** (a reference to the original data) whenever possible rather than a **copy**, it can lead to some unexpected bugs and performance bottlenecks.

Here is when you should think twice before using it:

---

## 1. When You Need to Protect the Original Data

The biggest "gotcha" with `ravel()` is that it returns a view. If you modify the flattened array, you are **simultaneously modifying the original multi-dimensional array.**

* **The Risk:** Accidental data corruption.
* **The Alternative:** Use `flatten()`. It always returns a copy, ensuring your original dataset remains untouched.

## 2. When Dealing with Non-Contiguous Memory

`ravel()` struggles to return a view if the array is "sliced" or has a complex memory layout (non-contiguous). In these cases, NumPy is forced to make a copy anyway, but it does so invisibly.

* **The Issue:** You lose the performance benefit of a view without the explicit intent of a copy.
* **The Alternative:** If you *need* a copy for safety, use `flatten()`. If you *need* a view for speed, check `.flags['C_CONTIGUOUS']` first.

## 3. When Memory Efficiency is Critical (and it fails)

If you are working with a massive dataset and you assume `ravel()` is just pointing to existing memory, but the array's "strides" make a view impossible, NumPy will suddenly double your memory usage by creating a hidden copy.

## 4. When Code Readability/Intent Matters

If you are writing a library or a shared script:

* Use `flatten()` to signal to other developers: *"I am creating a new, independent 1D array."*
* Use `ravel()` to signal: *"I am just reshaping this data temporarily for an operation and want to avoid memory overhead."*

---

### Comparison at a Glance

| Feature | `ravel()` | `flatten()` |
| --- | --- | --- |
| **Return Type** | View (usually) | Copy (always) |
| **Memory Impact** | Low (if view) | Higher (new allocation) |
| **Source Safety** | **Unsafe** (modifies original) | **Safe** |
| **Speed** | Faster (no allocation) | Slower |

---

**Would you like me to write a quick code snippet to demonstrate how `ravel` can accidentally change your original array's values?**


In [7]:
6. 
three_d = np.ones((3,3,3))
print(three_d)
print(three_d.shape)
print(three_d.ndim)
x = three_d.ravel()
print(x)


[[[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]]
(3, 3, 3)
3
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1.]


7. 
In NumPy, "memory sharing" refers to multiple array objects pointing to the same underlying block of RAM. When you use `ravel()`, NumPy attempts to create a **view** rather than a **copy**.

Instead of allocating new space for the data, NumPy simply creates a new "window" (the flattened array) that looks at the original memory addresses.

---

## How It Works Under the Hood

Every NumPy array has a `data` pointer, `shape`, and `strides`. When you call `ravel()`:

1. NumPy checks if the array's memory layout is **contiguous** (the data elements follow each other in a predictable sequence).
2. If it is, `ravel()` creates a new array object with a different `shape` (1D), but points it to the **exact same memory address** as the original.
3. If the original array is a "slice" or has a non-standard layout, memory sharing might be impossible, and NumPy will default to a copy.

---

## The "Danger" of Shared Memory

The primary implication of memory sharing is **mutability**. Because they share the same memory, changes are bidirectional.

### Example Scenario

Imagine a 2D matrix :


If you create , both  and  are looking at the same sequence in memory: `[1, 2, 3, 4]`.

* If you change 
* The original matrix  becomes:



---

## Checking for Shared Memory

If you aren't sure whether two arrays share the same memory, you can use the `np.shares_memory()` function. This is a crucial debugging step when working with large datasets where accidental mutations can ruin a model.

```python
import numpy as np

arr = np.array([[1, 2], [3, 4]])
raveled_arr = arr.ravel()

print(np.shares_memory(arr, raveled_arr)) # Returns: True

```

---

## Why Use It?

If you are working with a **10GB dataset**, `ravel()` is nearly instantaneous and uses almost zero additional RAM. `flatten()`, by comparison, would require another 10GB of RAM to create the copy, potentially crashing your system.

**Would you like me to show you an example of a "non-contiguous" array where `ravel()` is forced to fail and create a copy instead?**

In [8]:
8. 
an = np.diag([1,2,3])
an
x = an.ravel()
print(x)
y = x.reshape(3,3)
print(y)

[1 0 0 0 2 0 0 0 3]
[[1 0 0]
 [0 2 0]
 [0 0 3]]


In [9]:
9.
arr = np.array([[1, 2, 3],
                [4, 5, 6]])

flat = arr.ravel()

result = flat * 2
print(result)


[ 2  4  6  8 10 12]


In [ ]:
10.
# Analysts prefer ravel() + vectorization because it is:

# Faster :

# Loops run in Python, while NumPy operations run in C internally.


# Cleaner code :

# Loop code becomes messy with large matrices.

#  Memory efficient :

# ravel() returns a view when possible, avoiding extra memory allocation.